In [1]:
%pip install rioxarray -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
%matplotlib inline

import sys
import rasterio
import geopandas as gpd
import datacube
import numpy as np
import xarray as xr
import rioxarray as rxr
from rasterio.features import rasterize
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import contextily as ctx

from datacube.utils.cog import write_cog
from matplotlib import colors as mcolours

sys.path.insert(1, "../Tools/")
from dea_tools.plotting import display_map
from dea_tools.landcover import plot_land_cover
from dea_tools.dask import create_local_dask_cluster

In [3]:
client = create_local_dask_cluster(return_client=True)


/env/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44819 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/44819/status,
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/44819/status,Workers: 1
Total threads: 15,Total memory: 117.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40749,Workers: 1
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/44819/status,Total threads: 15
Started: Just now,Total memory: 117.21 GiB
Comm: tcp://127.0.0.1:40839,Total threads: 15
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/45885/status,Memory: 117.21 GiB
Nanny: tcp://127.0.0.1:38259,


In [4]:
dc = datacube.Datacube(app='lc_images_for_animation')

In [5]:
# Get coastlines vector data to later use for masking
coastlines_shp = 'tasmania/csttascd_r.shp'
coastlines_gdf = gpd.read_file(coastlines_shp).to_crs('EPSG:3577')
coastlines_gdf = coastlines_gdf.query("STATE_CODE != 0")
coastlines_gdf.crs

<Projected CRS: EPSG:3577>
Name: GDA94 / Australian Albers
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Australia - Australian Capital Territory; New South Wales; Northern Territory; Queensland; South Australia; Tasmania; Western Australia; Victoria.
- bounds: (112.85, -43.7, 153.69, -9.86)
Coordinate Operation:
- name: Australian Albers
- method: Albers Equal Area
Datum: Geocentric Datum of Australia 1994
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [6]:
coastlines_gdf.head()

,AREA,PERIMETER,CSTTASCD_,CSTTASCD_I,FEAT_CODE,ISLAND_NAM,GROUP_NAME,STATE_CODE,Q_INFO,UFI,geometry
1,5.923500e-07,0.003136,3,3,island,NORTH EAST ISLET,HOGAN GROUP,7,AE000899,AE00004399,"POLYGON ((1309902.153 -4363383.137, 1309894.52..."
2,1.143640e-05,0.019055,4,4,island,TWIN ISLETS,HOGAN GROUP,7,AE000898,AE00005381,"POLYGON ((1306652.098 -4363734.427, 1306579.12..."
3,4.184000e-07,0.002710,5,5,island,None,FORTY FOOT ROCKS,7,AE000881,AE00005114,"POLYGON ((1257620.317 -4357605.924, 1257590.79..."
4,6.144000e-07,0.003520,6,6,island,None,FORTY FOOT ROCKS,7,AE000881,AE00005111,"POLYGON ((1257721.085 -4357694.648, 1257711.76..."
5,3.202010e-05,0.028563,7,7,island,LONG ISLET,HOGAN GROUP,7,AE000898,AE00004835,"POLYGON ((1308032.752 -4363802.192, 1308181.97..."


In [7]:
#coastlines_gdf.explore()

In [8]:
LEVEL3_COLOUR_SCHEME = {
    111: (172, 188, 45, 255, "Cultivated terrestrial vegetation"),
    112: (14, 121, 18, 255, "Natural terrestrial vegetation"),
    124: (30, 191, 121, 255, "Natural aquatic vegetation"),
    215: (218, 92, 105, 255, "Artificial surface"),
    216: (243, 171, 105, 255, "Natural bare surface"),
    220: (77, 159, 220, 255, "Water"),
    255: (255, 255, 255, 255, "No Data"),
}

In [9]:
LEVEL4_COLOUR_SCHEME = {
               1: (151, 187, 26, 255, 'Cultivated Terrestrial\n Vegetated:'),
               2: (151, 187, 26, 255, 'Cultivated Terrestrial\n Vegetated: Woody'),
               3: (209, 224, 51, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous'),
               4: (197, 168, 71, 255, 'Cultivated Terrestrial\n Vegetated: Closed\n (> 65 %)'),
               5: (205, 181, 75, 255, 'Cultivated Terrestrial\n Vegetated: Open\n (40 to 65 %)'),
               6: (213, 193, 79, 255, 'Cultivated Terrestrial\n Vegetated: Open\n (15 to 40 %)'),
               7: (228, 210, 108, 255, 'Cultivated Terrestrial\n Vegetated: Sparse\n (4 to 15 %)'),
               8: (242, 227, 138, 255, 'Cultivated Terrestrial\n Vegetated: Scattered\n (1 to 4 %)'),
               9: (197, 168, 71, 255, 'Cultivated Terrestrial\n Vegetated: Woody Closed\n (> 65 %)'),
               10: (205, 181, 75, 255, 'Cultivated Terrestrial\n Vegetated: Woody Open\n (40 to 65 %)'),
               11: (213, 193, 79, 255, 'Cultivated Terrestrial\n Vegetated: Woody Open\n (15 to 40 %)'),
               12: (228, 210, 108, 255, 'Cultivated Terrestrial\n Vegetated: Woody Sparse\n (4 to 15 %)'),
               13: (242, 227, 138, 255, 'Cultivated Terrestrial\n Vegetated: Woody Scattered\n (1 to 4 %)'),
               14: (228, 224, 52, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Closed\n (> 65 %)'),
               15: (235, 232, 84, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Open\n (40 to 65 %)'),
               16: (242, 240, 127, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Open\n (15 to 40 %)'),
               17: (249, 247, 174, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Sparse\n (4 to 15 %)'),
               18: (255, 254, 222, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Scattered\n (1 to 4 %)'),
               19: (14, 121, 18, 255, 'Natural Terrestrial Vegetated:'),
               20: (26, 177, 87, 255, 'Natural Terrestrial Vegetated: Woody'),
               21: (94, 179, 31, 255, 'Natural Terrestrial Vegetated: Herbaceous'),
               22: (14, 121, 18, 255, 'Natural Terrestrial Vegetated: Closed (> 65 %)'),
               23: (45, 141, 47, 255, 'Natural Terrestrial Vegetated: Open (40 to 65 %)'),
               24: (80, 160, 82, 255, 'Natural Terrestrial Vegetated: Open (15 to 40 %)'),
               25: (117, 180, 118, 255, 'Natural Terrestrial Vegetated: Sparse (4 to 15 %)'),
               26: (154, 199, 156, 255, 'Natural Terrestrial Vegetated: Scattered (1 to 4 %)'),
               27: (14, 121, 18, 255, 'Natural Terrestrial Vegetated: Woody Closed (> 65 %)'),
               28: (45, 141, 47, 255, 'Natural Terrestrial Vegetated: Woody Open (40 to 65 %)'),
               29: (80, 160, 82, 255, 'Natural Terrestrial Vegetated: Woody Open (15 to 40 %)'),
               30: (117, 180, 118, 255, 'Natural Terrestrial Vegetated: Woody Sparse (4 to 15 %)'),
               31: (154, 199, 156, 255, 'Natural Terrestrial Vegetated: Woody Scattered (1 to 4 %)'),
               32: (119, 167, 30, 255, 'Natural Terrestrial Vegetated: Herbaceous Closed (> 65 %)'),
               33: (136, 182, 51, 255, 'Natural Terrestrial Vegetated: Herbaceous Open (40 to 65 %)'),
               34: (153, 196, 80, 255, 'Natural Terrestrial Vegetated: Herbaceous Open (15 to 40 %)'),
               35: (170, 212, 113, 255, 'Natural Terrestrial Vegetated: Herbaceous Sparse (4 to 15 %)'),
               36: (186, 226, 146, 255, 'Natural Terrestrial Vegetated: Herbaceous Scattered (1 to 4 %)'),
               37: (86, 236, 231, 255, 'Cultivated Aquatic Vegetated:'),
               38: (61, 170, 140, 255, 'Cultivated Aquatic Vegetated: Woody'),
               39: (82, 231, 172, 255, 'Cultivated Aquatic Vegetated: Herbaceous'),
               40: (43, 210, 203, 255, 'Cultivated Aquatic Vegetated: Closed (> 65 %)'),
               41: (73, 222, 216, 255, 'Cultivated Aquatic Vegetated: Open (40 to 65 %)'),
               42: (110, 233, 228, 255, 'Cultivated Aquatic Vegetated: Open (15 to 40 %)'),
               43: (149, 244, 240, 255, 'Cultivated Aquatic Vegetated: Sparse (4 to 15 %)'),
               44: (187, 255, 252, 255, 'Cultivated Aquatic Vegetated: Scattered (1 to 4 %)'),
               50: (82, 231, 196, 255, 'Cultivated Aquatic Vegetated: Herbaceous Closed (> 65 %)'),
               51: (113, 237, 208, 255, 'Cultivated Aquatic Vegetated: Herbaceous Open (40 to 65 %)'),
               52: (144, 243, 220, 255, 'Cultivated Aquatic Vegetated: Herbaceous Open (15 to 40 %)'),
               53: (175, 249, 232, 255, 'Cultivated Aquatic Vegetated: Herbaceous Sparse (4 to 15 %)'),
               54: (207, 255, 244, 255, 'Cultivated Aquatic Vegetated: Herbaceous Scattered (1 to 4 %)'),
               55: (30, 191, 121, 255, 'Natural Aquatic Vegetated:'),
               56: (18, 142, 148, 255, 'Natural Aquatic Vegetated: Woody'),
               57: (112, 234, 134, 255, 'Natural Aquatic Vegetated: Herbaceous'),
               58: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Closed (> 65 %)'),
               59: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Open (40 to 65 %)'),
               60: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Open (15 to 40 %)'),
               61: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Sparse (4 to 15 %)'),
               62: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Scattered (1 to 4 %)'),
               63: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Woody Closed (> 65 %)'),
               64: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Woody Closed (> 65 %) Water > 3 months (semi-) permenant'),
               65: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Woody Closed (> 65 %) Water < 3 months (temporary or seasonal)'),
               66: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Woody Open (40 to 65 %)'),
               67: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Woody Open (40 to 65 %) Water > 3 months (semi-) permenant'),
               68: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Woody Open (40 to 65 %) Water < 3 months (temporary or seasonal)'),
               69: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Woody Open (15 to 40 %)'),
               70: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Woody Open (15 to 40 %) Water > 3 months (semi-) permenant'),
               71: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Woody Open (15 to 40 %) Water < 3 months (temporary or seasonal)'),
               72: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Woody Sparse (4 to 15 %)'),
               73: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Woody Sparse (4 to 15 %) Water > 3 months (semi-) permenant'),
               74: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Woody Sparse (4 to 15 %) Water < 3 months (temporary or seasonal)'),
               75: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Woody Scattered (1 to 4 %)'),
               76: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Woody Scattered (1 to 4 %) Water > 3 months (semi-) permenant'),
               77: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Woody Scattered (1 to 4 %) Water < 3 months (temporary or seasonal)'),
               78: (39, 204, 139, 255, 'Natural Aquatic Vegetated: Herbaceous Closed (> 65 %)'),
               79: (39, 204, 139, 255, 'Natural Aquatic Vegetated: Herbaceous Closed (> 65 %) Water > 3 months (semi-) permenant'),
               80: (39, 204, 139, 255, 'Natural Aquatic Vegetated: Herbaceous Closed (> 65 %) Water < 3 months (temporary or seasonal)'),
               81: (66, 216, 159, 255, 'Natural Aquatic Vegetated: Herbaceous Open (40 to 65 %)'),
               82: (66, 216, 159, 255, 'Natural Aquatic Vegetated: Herbaceous Open (40 to 65 %) Water > 3 months (semi-) permenant'),
               83: (66, 216, 159, 255, 'Natural Aquatic Vegetated: Herbaceous Open (40 to 65 %) Water < 3 months (temporary or seasonal)'),
               84: (99, 227, 180, 255, 'Natural Aquatic Vegetated: Herbaceous Open (15 to 40 %)'),
               85: (99, 227, 180, 255, 'Natural Aquatic Vegetated: Herbaceous Open (15 to 40 %) Water > 3 months (semi-) permenant'),
               86: (99, 227, 180, 255, 'Natural Aquatic Vegetated: Herbaceous Open (15 to 40 %) Water < 3 months (temporary or seasonal)'),
               87: (135, 239, 201, 255, 'Natural Aquatic Vegetated: Herbaceous Sparse (4 to 15 %)'),
               88: (135, 239, 201, 255, 'Natural Aquatic Vegetated: Herbaceous Sparse (4 to 15 %) Water > 3 months (semi-) permenant'),
               89: (135, 239, 201, 255, 'Natural Aquatic Vegetated: Herbaceous Sparse (4 to 15 %) Water < 3 months (temporary or seasonal)'),
               90: (171, 250, 221, 255, 'Natural Aquatic Vegetated: Herbaceous Scattered (1 to 4 %)'),
               91: (171, 250, 221, 255, 'Natural Aquatic Vegetated: Herbaceous Scattered (1 to 4 %) Water > 3 months (semi-) permenant'),
               92: (171, 250, 221, 255, 'Natural Aquatic Vegetated: Herbaceous Scattered (1 to 4 %) Water < 3 months (temporary or seasonal)'),
               93: (218, 92, 105, 255, 'Artificial Surface:'),
               94: (243, 171, 105, 255, 'Natural Surface:'),
               95: (255, 230, 140, 255, 'Natural Surface: Sparsely vegetated'),
               96: (250, 210, 110, 255, 'Natural Surface: Very sparsely vegetated'),
               97: (243, 171, 105, 255, 'Natural Surface: Bare areas, unvegetated'),
               98: (77, 159, 220, 255, 'Water:'),
               99: (77, 159, 220, 255, 'Water: (Water)'),
               100: (187, 220, 233, 255, 'Water: (Water) Tidal area'),
               101: (27, 85, 186, 255, 'Water: (Water) Perennial (> 9 months)'),
               102: (52, 121, 201, 255, 'Water: (Water) Non-perennial (7 to 9 months)'),
               103: (79, 157, 217, 255, 'Water: (Water) Non-perennial (4 to 6 months)'),
               104: (133, 202, 253, 255, 'Water: (Water) Non-perennial (1 to 3 months)'),
               255: (255, 255, 255, 255, "No Data")
               }

In [10]:
lat_range = (-40.08, -43.8)
lon_range = (144.33, 148.72)
time = ('2020','2023')

In [11]:
#display_map(x=lon_range, y=lat_range)

In [12]:
query = {
    'x': lon_range,
    'y': lat_range,
    'time':time
}

lazy_ds = dc.load(product='ga_ls_landcover_class_cyear_3',
                 measurements='level4',
                 output_crs='EPSG:3577',
                dask_chunks={"time": 1, "x": 1024, "y": 1024},
                resolution=(-300, 300),
                 **query)

In [13]:
bathy_ds = dc.load(
    product='ga_ausbathytopo250m_2023',
    output_crs='EPSG:3577',
    resolution=(-300,300),
    **query
)

In [14]:
bathy_ds


<xarray.Dataset> Size: 8MB
Dimensions:       (time: 1, y: 1475, x: 1390)
Coordinates:
  * time          (time) datetime64[ns] 8B 2023-07-02T11:59:59.999999
  * y             (y) float64 12kB -4.434e+06 -4.435e+06 ... -4.877e+06
  * x             (x) float64 11kB 1.028e+06 1.028e+06 ... 1.444e+06 1.445e+06
    spatial_ref   int32 4B 3577
Data variables:
    height_depth  (time, y, x) float32 8MB 68.0 70.0 ... -3.217e+03 -3.218e+03
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [15]:
lazy_ds

<xarray.Dataset> Size: 8MB
Dimensions:      (time: 4, y: 1475, x: 1390)
Coordinates:
  * time         (time) datetime64[ns] 32B 2020-07-01T23:59:59.999999 ... 202...
  * y            (y) float64 12kB -4.434e+06 -4.435e+06 ... -4.877e+06
  * x            (x) float64 11kB 1.028e+06 1.028e+06 ... 1.444e+06 1.445e+06
    spatial_ref  int32 4B 3577
Data variables:
    level4       (time, y, x) uint8 8MB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [16]:
# Convert the polygons to a mask
shapes = [(geom, 1) for geom in coastlines_gdf.geometry]
mask = rasterize(
    shapes,
    out_shape=lazy_ds.rio.shape,
    transform=lazy_ds.rio.transform(),
    fill=0,
    dtype='uint8'
)
masked_ds = lazy_ds.where(mask)

In [17]:
# only keep bathy data from coastlines out

masked_bathy = bathy_ds.where(bathy_ds < 0)

In [18]:
# def plot_layer(colours, data, data_2):
#     # Ensure additional_data is a DataArray
#     if isinstance(data_2, xr.Dataset):
#         data_2 = data_2.to_array().isel(variable=0)
    
#     minx, miny, maxx, maxy = data.rio.bounds()
#     colour_arr = []
#     for key, value in colours.items():
#         colour_arr.append(np.array(value[:-1]) / 255)
 
#     cmap = mcolours.ListedColormap(colour_arr)
#     bounds = list(colours)
#     bounds.append(256)  # Add upper bound to make sure highest value (255) included in last colour bin
#     norm = mcolours.BoundaryNorm(np.array(bounds) - 0.1, cmap.N)
#     labels = {'ticks': [111, 112, 124, 215, 216, 220, 255]}

#     if len(data.time) == 1: 
#         # Plot the provided layer for a single time step
#         fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={'projection': ccrs.epsg(3577)})
#         data_2.plot(ax=ax, cmap='Blues_r', add_colorbar=False, transform=ccrs.epsg(3577))
#         im = data.isel(time=0).plot(ax=ax,
#                                     cmap=cmap, 
#                                     norm=norm, 
#                                     add_colorbar=False, 
#                                     transform=ccrs.epsg(3577))
#         ax.set_extent([minx, maxx, miny, maxy], crs=ccrs.epsg(3577))
#         ctx.add_basemap(ax, crs=ccrs.epsg(3577).proj4_init, source=ctx.providers.OpenStreetMap.Mapnik)

#         #ax.gridlines(draw_labels=True)
#         year = str(time.dt.year.values)
#         ax.set_title(f'Time: {year}')

#     else:
#         num_times = len(data.time)
#         fig, axes = plt.subplots(nrows=num_times, figsize=(5, 5 * num_times), subplot_kw={'projection': ccrs.epsg(3577)})
#         for i, time in enumerate(data.time):
#             ax = axes[i] if num_times > 1 else axes
#             data_2.plot(ax=ax, cmap='Blues_r', add_colorbar=False, transform=ccrs.epsg(3577))
#             im = data.sel(time=time).plot(
#                 ax=ax, cmap=cmap, norm=norm, add_colorbar=False, transform=ccrs.epsg(3577)
#             )
            
#             # Add a basemap
#             ax.set_extent([minx, maxx, miny, maxy], crs=ccrs.epsg(3577))
#             ctx.add_basemap(ax, crs=ccrs.epsg(3857).proj4_init, source=ctx.providers.OpenStreetMap.Mapnik)
            
#             #ax.gridlines(draw_labels=True)
#             year = str(time.dt.year.values)
#             ax.set_title(f'{year}')

#     return im


In [19]:
def plot_layer_individual_images(colours, data, data_2):
    # Ensure additional_data is a DataArray
    if isinstance(data_2, xr.Dataset):
        data_2 = data_2.to_array().isel(variable=0)
    
    minx, miny, maxx, maxy = data.rio.bounds()
    colour_arr = []
    for key, value in colours.items():
        colour_arr.append(np.array(value[:-1]) / 255)
 
    cmap = mcolours.ListedColormap(colour_arr)
    bounds = list(colours)
    bounds.append(256)  # Add upper bound to make sure highest value (255) included in last colour bin
    norm = mcolours.BoundaryNorm(np.array(bounds) - 0.1, cmap.N)

    for time in data.time:
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.epsg(3577)})
        data_2.plot(ax=ax, cmap='Blues_r', add_colorbar=False, transform=ccrs.epsg(3577))
        # im = data.sel(time=time).plot(
        #     ax=ax, cmap=cmap, norm=norm, add_colorbar=False, transform=ccrs.epsg(3577)
        # )
        
        # Add a basemap
        ax.set_title('')
        ax.set_extent([minx, maxx, miny, maxy], crs=ccrs.epsg(3577))
        ctx.add_basemap(ax, crs=ccrs.epsg(3577).proj4_init, source=ctx.providers.OpenStreetMap.Mapnik)
        
        #ax.gridlines(draw_labels=True)
        year = str(time.dt.year.values)
        #ax.set_title(f'Time: {year}')
        ax.text(0.5, 0.98, f'Time: {year}', transform=ax.transAxes, ha='center', va='top', fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
        plt.savefig(f'outputs/{year}_tassie_test_landcover_level4.png', bbox_inches='tight')
        plt.close(fig)

    #return im


In [20]:
#minx, miny, maxx, maxy = masked_ds.rio.bounds()

#plot_layer(LEVEL4_COLOUR_SCHEME, masked_ds.level4, bathy_ds)
plot_layer_individual_images(LEVEL4_COLOUR_SCHEME, masked_ds.level4, masked_bathy)